<a href="https://colab.research.google.com/github/EliteView/EliteView/blob/main/titanic_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **My First Ml project**

#**Loading Data**

In [89]:
import pandas as pd
file = "https://raw.githubusercontent.com/datasciencedojo/datasets/refs/heads/master/titanic.csv"
df = pd.read_csv(file)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [91]:
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
#df['Cabin'] = df['Cabin'].fillna(df['Cabin'].mean())

df['Cabin'] = pd.to_numeric(df['Cabin'], errors='coerce')
df['Cabin'].fillna(df['Cabin'].mode())

df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])


In [92]:
df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


In [93]:
df.tail()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
886,0,2,male,27.000000,0,0,13.00,S
887,1,1,female,19.000000,0,0,30.00,S
888,0,3,female,29.699118,1,2,23.45,S
889,1,1,male,26.000000,0,0,30.00,C
890,0,3,male,32.000000,0,0,7.75,Q


## **Data Preparation**


## **Data Preparation as X and Y**

In [94]:
y = df['Survived']
x = df.drop('Survived', axis=1)
x

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.000000,1,0,7.2500,S
1,1,female,38.000000,1,0,71.2833,C
2,3,female,26.000000,0,0,7.9250,S
3,1,female,35.000000,1,0,53.1000,S
4,3,male,35.000000,0,0,8.0500,S
...,...,...,...,...,...,...,...
886,2,male,27.000000,0,0,13.0000,S
887,1,female,19.000000,0,0,30.0000,S
888,3,female,29.699118,1,2,23.4500,S
889,1,male,26.000000,0,0,30.0000,C


## **Data Separation**

## **Data Splitting**

In [95]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test =train_test_split(x, y, test_size=0.15, random_state=1)

In [96]:
x_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
85,3,female,33.000000,3,0,15.8500,S
195,1,female,58.000000,0,0,146.5208,C
585,1,female,18.000000,0,2,79.6500,S
386,3,male,1.000000,5,2,46.9000,S
371,3,male,18.000000,1,0,6.4958,S
...,...,...,...,...,...,...,...
715,3,male,19.000000,0,0,7.6500,S
767,3,female,30.500000,0,0,7.7500,Q
72,2,male,21.000000,0,0,73.5000,S
235,3,female,29.699118,0,0,7.5500,S


## **Preprocessing steps**

In [97]:
from sklearn.impute import SimpleImputer
numeric_features = ["Age", "SibSp", "Parch", "Fare"]
numeric_transformer = SimpleImputer(strategy="median")

In [98]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_features = ["Pclass", "Sex", "Embarked"]
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [99]:

from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


## **Model pipeline**

In [100]:
from sklearn.ensemble import RandomForestClassifier
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

In [101]:

# Train model
clf.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Pclass', 'Sex',
                                                   'Embarked'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [103]:
# Evaluate
print("Test accuracy:", clf.score(x_test, y_test))

Test accuracy: 0.7835820895522388
